# VAE Training Notebook

This notebook documents the training procedure for the pre-trained models
used in the VAE lesson §8 visualization. The actual training was performed
using a Node.js port of this logic (see `scripts/train-vae.cjs`).

## Dataset
- 1000 points from a 2D four-cluster Gaussian mixture
- Cluster centers: `(±2, ±2)`
- Cluster std: `0.2`

## Architecture
- `TinyVAE`: data_dim=2, latent_dim=2, hidden_dim=16
- Encoder: `Linear(2→16) → tanh → Linear(16→2)[μ] + Linear(16→2)[log σ]`
- Decoder: `Linear(2→16) → tanh → Linear(16→2)`

## β values trained
`[0.25, 0.5, 1.0, 2.0, 5.0, 10.0]`

In [ ]:
import numpy as np
import json
import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
    print('PyTorch not available — see scripts/train-vae.cjs for the Node.js equivalent.')

print(f'PyTorch available: {HAS_TORCH}')

In [ ]:
# Dataset generation
np.random.seed(42)
centers = np.array([[2, 2], [2, -2], [-2, 2], [-2, -2]], dtype=np.float32)
N_PER_CLUSTER = 250

X = np.vstack([
    np.random.normal(c, 0.2, (N_PER_CLUSTER, 2))
    for c in centers
]).astype(np.float32)
labels = np.repeat(np.arange(4), N_PER_CLUSTER)

fig, ax = plt.subplots(figsize=(5, 5))
colors = ['#c0392b', '#2c5f8d', '#5a8a6a', '#d4a437']
for ci in range(4):
    mask = labels == ci
    ax.scatter(X[mask, 0], X[mask, 1], c=colors[ci], s=6, alpha=0.5)
ax.set_title('Training data — 4-cluster mixture')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()
print(f'Dataset: {len(X)} points in 4 clusters')

In [ ]:
if HAS_TORCH:
    import torch
    import torch.nn as nn

    class TinyVAE(nn.Module):
        def __init__(self, beta=1.0):
            super().__init__()
            self.enc1 = nn.Linear(2, 16)
            self.enc_mu = nn.Linear(16, 2)
            self.enc_logsigma = nn.Linear(16, 2)
            self.dec1 = nn.Linear(2, 16)
            self.dec2 = nn.Linear(16, 2)
            self.beta = beta

        def encode(self, x):
            h = torch.tanh(self.enc1(x))
            return self.enc_mu(h), self.enc_logsigma(h)

        def reparam(self, mu, log_sigma):
            return mu + torch.exp(log_sigma) * torch.randn_like(mu)

        def decode(self, z):
            h = torch.tanh(self.dec1(z))
            return self.dec2(h)

        def loss(self, x):
            mu, ls = self.encode(x)
            z = self.reparam(mu, ls)
            x_hat = self.decode(z)
            recon = ((x - x_hat) ** 2).sum(dim=1).mean()
            kl = 0.5 * (torch.exp(2*ls) + mu**2 - 1 - 2*ls).sum(dim=1).mean()
            return recon + self.beta * kl, recon.item(), kl.item()

    print('TinyVAE defined')
else:
    print('Skipping — run scripts/train-vae.cjs instead')

In [ ]:
# Training loop (requires PyTorch)
if HAS_TORCH:
    BETAS = [0.25, 0.5, 1.0, 2.0, 5.0, 10.0]
    EPOCHS = 5000
    weights_out = {}

    Xt = torch.tensor(X)

    for beta in BETAS:
        torch.manual_seed(1337 + int(beta * 100))
        model = TinyVAE(beta=beta)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3)

        for epoch in range(EPOCHS):
            idx = np.random.choice(len(Xt), 64, replace=False)
            loss, _, _ = model.loss(Xt[idx])
            opt.zero_grad(); loss.backward(); opt.step()
            if (epoch + 1) % 1000 == 0:
                print(f'  beta={beta}, epoch={epoch+1}, loss={loss.item():.4f}')

        weights_out[f'beta_{beta}'] = {
            k: v.detach().numpy().tolist()
            for k, v in model.state_dict().items()
        }

    weights_out['metadata'] = {
        'centers': centers.tolist(),
        'n_per_cluster': N_PER_CLUSTER,
        'latent_dim': 2,
        'hidden_dim': 16,
        'epochs': EPOCHS,
    }

    with open('../src/lessons/vae/assets/vae-weights.json', 'w') as f:
        json.dump(weights_out, f)

    print('Weights saved!')
else:
    print('Training requires PyTorch. Use scripts/train-vae.cjs instead.')

In [ ]:
# Visual inspection — load and plot latent spaces
# (Works regardless of whether training was done here or via Node.js)

with open('../src/lessons/vae/assets/vae-weights.json') as f:
    saved = json.load(f)

def encode_np(weights, X_arr):
    """NumPy forward pass through the encoder — outputs mu."""
    W1 = np.array(weights['enc1.weight'])   # (16, 2)
    b1 = np.array(weights['enc1.bias'])     # (16,)
    Wmu = np.array(weights['enc_mu.weight'])  # (2, 16)
    bmu = np.array(weights['enc_mu.bias'])    # (2,)
    h = np.tanh(X_arr @ W1.T + b1)
    mu = h @ Wmu.T + bmu
    return mu

betas_plot = [0.25, 1.0, 10.0]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, beta in zip(axes, betas_plot):
    key = f'beta_{beta}'
    if key not in saved:
        ax.set_title(f'β={beta} not found')
        continue
    mu = encode_np(saved[key], X)
    for ci in range(4):
        mask = labels == ci
        ax.scatter(mu[mask, 0], mu[mask, 1], c=colors[ci], s=4, alpha=0.4)
    # Prior 1σ circle
    th = np.linspace(0, 2*np.pi, 100)
    ax.plot(np.cos(th), np.sin(th), 'k--', lw=0.8, alpha=0.3)
    ax.set_title(f'β = {beta}', fontsize=12)
    ax.set_aspect('equal')
    ax.axhline(0, c='k', lw=0.4, alpha=0.3)
    ax.axvline(0, c='k', lw=0.4, alpha=0.3)

fig.suptitle('Encoder means in latent space', fontsize=13)
plt.tight_layout()
plt.show()

# Confirm expected behavior
for beta in betas_plot:
    key = f'beta_{beta}'
    if key not in saved: continue
    mu = encode_np(saved[key], X)
    # cluster spread = mean of per-cluster-mean norms
    spread = np.mean([
        np.linalg.norm(mu[labels==ci].mean(axis=0))
        for ci in range(4)
    ])
    print(f'β={beta:5.2f}: cluster spread = {spread:.3f}')